# checkout_v2 — A/B test analysis

Reads the tables built by the SQL scripts in `sql/` from BigQuery and produces the
figures used in the README and in `memo.md`.

Run the SQL files `01` → `04` in BigQuery first. Authentication (once):
`gcloud auth application-default login`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery

PROJECT = "ab-test-checkout"   # your GCP project id
DATASET = "ab_test"
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

client = bigquery.Client(project=PROJECT)

def q(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

In [ ]:
# ---- plotting style --------------------------------------------------------
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
PLATFORM_COLORS = {"ios": "#2a78d6", "android": "#eb6834", "web": "#1baf7a"}
NEUTRAL = "#52514e"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK2, "ytick.color": INK2, "font.size": 10,
    "lines.linewidth": 2, "savefig.dpi": 160, "savefig.bbox": "tight",
})

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    plt.show()

## 1. Sample Ratio Mismatch

Share of users assigned to treatment, by platform and day. Under correct randomisation
every line fluctuates around 0.5.

In [ ]:
srm = q(f'''
  SELECT u.platform, DATE(a.assigned_ts) AS assigned_date,
         COUNT(*) AS n,
         COUNTIF(a.variant = 'treatment') / COUNT(*) AS share_treatment
  FROM {DATASET}.assignments a JOIN {DATASET}.users u USING (user_id)
  GROUP BY 1, 2 ORDER BY 1, 2
''')
srm["assigned_date"] = pd.to_datetime(srm["assigned_date"])

In [ ]:
def plot_srm(srm):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axhline(0.5, color=INK2, linewidth=1, linestyle="--")
    bug = srm[(srm.platform == "android")
              & srm.assigned_date.between("2026-09-08", "2026-09-10")]
    ax.axvspan(bug.assigned_date.min() - pd.Timedelta(hours=12),
               bug.assigned_date.max() + pd.Timedelta(hours=12),
               color="#eb6834", alpha=0.08, linewidth=0)
    for p, d in srm.groupby("platform"):
        ax.plot(d.assigned_date, d.share_treatment, color=PLATFORM_COLORS[p],
                marker="o", markersize=3.5, label=p)
        last = d.iloc[-1]
        ax.annotate(p, (last.assigned_date, last.share_treatment), xytext=(6, 0),
                    textcoords="offset points", va="center", color=INK2)
    ax.annotate("logging bug:\nandroid, 8–10 Sep", (bug.assigned_date.min(), bug.share_treatment.min()),
                xytext=(40, -6), textcoords="offset points", color=INK, fontsize=9,
                arrowprops=dict(arrowstyle="-", color=INK2, linewidth=0.8))
    ax.set_ylim(0.3, 0.65)
    ax.set_ylabel("share in treatment")
    ax.set_title("Treatment share by platform and assignment day")
    ax.legend(frameon=False, loc="upper right", ncol=3)
    fig.autofmt_xdate()
    return fig

save(plot_srm(srm), "01_srm")

## 2. Main results

Relative lift (treatment vs control) with 95% confidence interval, on the clean
population (bugged cell removed in both variants). CUPED uses pre-period sessions.

In [ ]:
res = q(f"SELECT * FROM {DATASET}.results")
res["rel_low"]  = 100 * res.ci95_low  / res.mean_control
res["rel_high"] = 100 * res.ci95_high / res.mean_control
res[res.segment == "all"][["metric", "mean_control", "mean_treatment", "rel_lift_pct",
                           "rel_low", "rel_high", "p_value", "se"]].round(4)

In [ ]:
LABELS = {
    "conversion": "Conversion", "conversion_cuped": "Conversion (CUPED)",
    "revenue_per_user": "Revenue / user", "revenue_per_user_cuped": "Revenue / user (CUPED)",
    "sessions_per_user": "Sessions / user (guardrail)",
}

def forest(df, label_col, title, colors=None):
    df = df.reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(df) + 1.2))
    ax.axvline(0, color=INK2, linewidth=1)
    for i, r in df.iterrows():
        y = len(df) - 1 - i
        c = (colors or {}).get(r[label_col], NEUTRAL)
        ax.plot([r.rel_low, r.rel_high], [y, y], color=c, linewidth=2, solid_capstyle="round")
        ax.plot(r.rel_lift_pct, y, "o", color=c, markersize=8,
                markeredgecolor=SURFACE, markeredgewidth=2)
        ax.annotate(f"{r.rel_lift_pct:+.1f}%  (p={r.p_value:.3f})", (r.rel_high, y),
                    xytext=(8, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels([LABELS.get(v, v) for v in df[label_col][::-1]])
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("relative lift vs control (%), 95% CI")
    ax.set_title(title)
    lo, hi = df.rel_low.min(), df.rel_high.max()
    ax.set_xlim(min(lo, 0) - 1, hi + (hi - min(lo, 0)) * 0.55)
    return fig

order = list(LABELS)
main = res[res.segment == "all"].set_index("metric").loc[order].reset_index()
save(forest(main, "metric", "checkout_v2 vs control — all users"), "02_main_results")

## 3. Segments

Primary metric (CUPED conversion) by platform. Three tests → Bonferroni threshold
0.05 / 3 ≈ 0.0167.

In [ ]:
seg = (res[(res.metric == "conversion_cuped") & (res.segment != "all")]
       .set_index("segment").loc[["ios", "android", "web"]].reset_index())
seg["significant_bonferroni"] = seg.p_value < 0.05 / 3
display(seg[["segment", "rel_lift_pct", "rel_low", "rel_high", "p_value",
             "significant_bonferroni"]].round(4))
save(forest(seg, "segment", "Conversion lift by platform (CUPED)", PLATFORM_COLORS),
     "03_segments")

## 4. Novelty effect

P(purchase | checkout started) by week since assignment (week 3 includes week 4).

In [ ]:
novelty_sql = Path("sql/04_novelty.sql").read_text()
nov = q(novelty_sql)
nov

In [ ]:
def plot_novelty(nov):
    fig, ax = plt.subplots(figsize=(7, 3.8))
    labels = ["week 1", "week 2", "week 3+"][: len(nov)]
    bars = ax.bar(labels, nov.rel_lift_pct, width=0.5, color=NEUTRAL, zorder=3)
    bars[0].set_color("#eb6834")
    for b, v in zip(bars, nov.rel_lift_pct):
        ax.annotate(f"+{v:.1f}%", (b.get_x() + b.get_width() / 2, v), xytext=(0, 4),
                    textcoords="offset points", ha="center", color=INK, fontsize=10)
    stable = nov.rel_lift_pct.iloc[1:].mean()
    ax.text(0, 1.02, f"Week 1 is inflated by novelty; from week 2 the lift settles at ≈ +{stable:.1f}%",
            transform=ax.transAxes, color=INK2, fontsize=9)
    ax.set_ylabel("relative lift (%)")
    ax.set_title("Checkout → purchase lift fades after week 1", pad=22)
    ax.grid(axis="x", visible=False)
    ax.set_ylim(0, nov.rel_lift_pct.max() * 1.2)
    return fig

save(plot_novelty(nov), "04_novelty")

## 5. Validation against the ground truth

The data is simulated (`generate_data.py`), so the analysis can be checked against
the effects that were injected:

| Injected | Recovered |
|---|---|
| Logging bug on android treatment, 8–10 Sep | SRM p < 0.001, localised to exactly those cells |
| Lift on checkout→purchase: +7% iOS/android, +1% web | iOS strongest; android and web not separable at this sample size (segments are noisy) |
| Novelty boost decaying over ~4 days | week-1 lift ≈ 2× the later weeks |
| AOV −1% | AOV flat (≈ €43.8 in both): the revenue gain comes from more orders, not bigger baskets |
| No effect on engagement | sessions/user not significant |